# Build a Job-Listing Aggregator — runnable notebook

This notebook is a Colab/Kaggle/Binder-friendly version of the **Build a Job-Listing Aggregator** project from the
[Python & Data Analysis course](https://github.com/abderrahim-lectures/python-data-analysis-course). It mirrors the
real, runnable example at [`examples/job-aggregator`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/job-aggregator) —
the only difference is the three sample "job board" HTML pages are embedded as strings below instead of separate
files, so the whole thing runs top to bottom in one notebook with no local file server needed.

See the full lesson (linked from the course's Real-World Projects page) for the step-by-step walkthrough of what
each part below does and why.

In [ ]:
!pip install beautifulsoup4 pandas

## The sample data

Three tiny sample "job board" pages, each with deliberately different HTML — a card layout, a bulleted list, and a
plain table — the same way three real job boards would never agree on markup. Two listings (a Senior Python Developer
role at Northwind Analytics, and a Data Analyst role at Contoso Retail) are posted on more than one board on purpose,
so there's something real to dedupe in Step 3 below.

In [ ]:
BOARD_ALPHA_HTML = '<!DOCTYPE html>\n<html lang="en">\n<head><meta charset="utf-8"><title>Alpha Jobs — Listings</title></head>\n<body>\n<h1>Alpha Jobs</h1>\n<div class="job-card" data-job-id="a-101">\n  <h2 class="job-title">Senior Python Developer</h2>\n  <span class="company">Northwind Analytics</span>\n  <span class="location">Remote</span>\n  <p class="description">Build and maintain data pipelines in Python. Experience with pandas and SQL required.</p>\n</div>\n<div class="job-card" data-job-id="a-102">\n  <h2 class="job-title">Data Analyst</h2>\n  <span class="company">Contoso Retail</span>\n  <span class="location">Chicago, IL</span>\n  <p class="description">Analyze sales data and build dashboards for the merchandising team.</p>\n</div>\n<div class="job-card" data-job-id="a-103">\n  <h2 class="job-title">Junior Backend Engineer</h2>\n  <span class="company">Fabrikam Health</span>\n  <span class="location">Remote</span>\n  <p class="description">Work on our patient-scheduling API. Python and Django experience a plus.</p>\n</div>\n<div class="job-card" data-job-id="a-104">\n  <h2 class="job-title">Machine Learning Engineer</h2>\n  <span class="company">Northwind Analytics</span>\n  <span class="location">Austin, TX</span>\n  <p class="description">Train and deploy ML models for demand forecasting. Python, scikit-learn, and MLOps experience wanted.</p>\n</div>\n</body>\n</html>\n'

BOARD_BETA_HTML = '<!DOCTYPE html>\n<html lang="en">\n<head><meta charset="utf-8"><title>Beta Careers — Open Roles</title></head>\n<body>\n<h1>Beta Careers</h1>\n<ul class="listings">\n  <li class="listing" data-ref="b-201">\n    <a class="position-title" href="/roles/b-201">Senior Python Developer</a>\n    <div class="employer">Northwind Analytics</div>\n    <div class="loc">Remote</div>\n    <div class="summary">We need a senior Python engineer to build and maintain our data pipelines. pandas + SQL required.</div>\n  </li>\n  <li class="listing" data-ref="b-202">\n    <a class="position-title" href="/roles/b-202">Frontend Engineer</a>\n    <div class="employer">Initech Software</div>\n    <div class="loc">New York, NY</div>\n    <div class="summary">Build customer-facing dashboards in React and TypeScript.</div>\n  </li>\n  <li class="listing" data-ref="b-203">\n    <a class="position-title" href="/roles/b-203">Python Automation Engineer</a>\n    <div class="employer">Globex Logistics</div>\n    <div class="loc">Remote</div>\n    <div class="summary">Write Python scripts to automate warehouse reporting and alerting.</div>\n  </li>\n</ul>\n</body>\n</html>\n'

BOARD_GAMMA_HTML = '<!DOCTYPE html>\n<html lang="en">\n<head><meta charset="utf-8"><title>Gamma Talent — Jobs</title></head>\n<body>\n<h1>Gamma Talent</h1>\n<table class="job-table">\n  <tbody>\n    <tr class="job-row" id="g-301">\n      <td class="col-title">Data Analyst</td>\n      <td class="col-company">Contoso Retail</td>\n      <td class="col-location">Chicago, IL</td>\n      <td class="col-desc">Build dashboards and reports for the merchandising team from sales data.</td>\n    </tr>\n    <tr class="job-row" id="g-302">\n      <td class="col-title">DevOps Engineer</td>\n      <td class="col-company">Umbrella Cloud</td>\n      <td class="col-location">Remote</td>\n      <td class="col-desc">Manage CI/CD pipelines and Kubernetes clusters. Python scripting is a plus.</td>\n    </tr>\n    <tr class="job-row" id="g-303">\n      <td class="col-title">Python Developer</td>\n      <td class="col-company">Wayne Fintech</td>\n      <td class="col-location">Remote</td>\n      <td class="col-desc">Build internal tools with Python and FastAPI for the risk team.</td>\n    </tr>\n  </tbody>\n</table>\n</body>\n</html>\n'

## Step 1-2: Parse each board and combine

One small parser function per board — each returns the same shape of dict (`title`, `company`, `location`,
`description`, `source`) no matter how different the underlying HTML is.

In [ ]:
from bs4 import BeautifulSoup


def parse_board_alpha(html):
    soup = BeautifulSoup(html, "html.parser")
    listings = []
    for card in soup.find_all("div", class_="job-card"):
        listings.append({
            "title": card.find("h2", class_="job-title").get_text(strip=True),
            "company": card.find("span", class_="company").get_text(strip=True),
            "location": card.find("span", class_="location").get_text(strip=True),
            "description": card.find("p", class_="description").get_text(strip=True),
            "source": "board_alpha",
        })
    return listings


def parse_board_beta(html):
    soup = BeautifulSoup(html, "html.parser")
    listings = []
    for item in soup.find_all("li", class_="listing"):
        listings.append({
            "title": item.find("a", class_="position-title").get_text(strip=True),
            "company": item.find("div", class_="employer").get_text(strip=True),
            "location": item.find("div", class_="loc").get_text(strip=True),
            "description": item.find("div", class_="summary").get_text(strip=True),
            "source": "board_beta",
        })
    return listings


def parse_board_gamma(html):
    soup = BeautifulSoup(html, "html.parser")
    listings = []
    for row in soup.find_all("tr", class_="job-row"):
        cells = row.find_all("td")
        listings.append({
            "title": cells[0].get_text(strip=True),
            "company": cells[1].get_text(strip=True),
            "location": cells[2].get_text(strip=True),
            "description": cells[3].get_text(strip=True),
            "source": "board_gamma",
        })
    return listings


all_listings = (
    parse_board_alpha(BOARD_ALPHA_HTML)
    + parse_board_beta(BOARD_BETA_HTML)
    + parse_board_gamma(BOARD_GAMMA_HTML)
)
print(f"Parsed {len(all_listings)} raw listings from 3 boards")

## Step 3: Dedupe with pandas

The same job posted to two boards will carry the same title and company text, so a normalized `title|company` hash
is a more reliable dedupe key here than hashing the whole row (the description wording can differ slightly board to
board, as it does in this sample data).

In [ ]:
import hashlib
import re

import pandas as pd


def dedupe_key(listing):
    normalized = f"{listing['title'].strip().lower()}|{listing['company'].strip().lower()}"
    normalized = re.sub(r"\s+", " ", normalized)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()[:16]


for listing in all_listings:
    listing["dedupe_key"] = dedupe_key(listing)

df = pd.DataFrame(all_listings)
before = len(df)
df = df.drop_duplicates(subset="dedupe_key", keep="first").reset_index(drop=True)
print(f"Deduped {before} listings -> {len(df)} unique jobs ({before - len(df)} duplicate posting(s) removed)")
df

## Step 4: Filter by keyword and alert on new matches

Filter to listings whose title or description mentions a keyword, then diff against a `seen` set of dedupe keys
already alerted on in a previous run, so re-running this cell against the same data reports zero new matches --
exactly what a real scheduled aggregator should do.

In [ ]:
KEYWORDS = ["python"]

pattern = "|".join(KEYWORDS)
text = df["title"].str.cat(df["description"], sep=" ")
matches = df[text.str.contains(pattern, case=False, regex=True, na=False)]
print(f"{len(matches)} unique listing(s) match keywords {KEYWORDS}")

# In a real run this `seen` set would be loaded from and saved back to a
# seen.json file (see filter_alerts.py) so it persists between runs. Here,
# starting it empty means every match below prints as "new" on this first run.
seen = set()
new_matches = matches[~matches["dedupe_key"].isin(seen)]

print(f"\n{len(new_matches)} NEW match(es):\n")
for _, row in new_matches.iterrows():
    print(f"- {row['title']} @ {row['company']} ({row['location']}) [{row['source']}]")

seen |= set(matches["dedupe_key"])
new_matches

## Where to go from here

This notebook stops at printing new matches. The real project (see the lesson) persists `seen` to a `seen.json` file
between runs, and the natural next step beyond that is wiring up a real notification -- an email via `smtplib`, or a
webhook to a Discord/Slack channel -- and scheduling the whole pipeline to run periodically instead of by hand.